# Story 3: channels

> **Project retired on 2026-09-30.** Findings and decision: [`project-conclusion.md`](../../docs/analysis/project-conclusion.md). Story 1 found no cell-level association to decompose (B3), so nothing beyond M1 was run.

Step 4 of the analysis plan ([`final-analysis-plan.md`](../../docs/analysis/final-analysis-plan.md) §6). **Question:** which mechanisms could produce the
association of Story 1?

| Order | Notebook | Question |
|---|---|---|
| 1 | `01_pre_analysis.ipynb` | What do the stories rely on? No outcome-on-treatment regressions. |
| 2 | `02_baseline.ipynb` | Story 1: is the lights–night-LST association robust enough to be worth explaining? |
| 3 | `03_iv.ipynb`, `03b_iv_robustness.ipynb` | Story 2: can the mining instrument identify the effect, and what does its null rule out? |
| 4 | `04_channels.ipynb` | Story 3: which mechanisms could produce the association? |

| Block | Content | ◇ Rule (plan §6) |
|---|---|---|
| M1 | PM2.5 channel accounting β_short − β_long = γ·δ, night and day; the IV with and without PM2.5 | γ ≤ 0 → the control cannot move β · δ ≥ 0 → premise fails · \|γδ\| < ¼\|β\| → PM2.5 explains little · night more negative than day → pollution unlikely the sole channel |
| later | land-cover accounting (Hansen, ESA CCI fractions, Duveiller et al. 2018 sensitivities), MAIAC AOD, NDVI/ET | only if Story 1 finds a robust association |

A causal decomposition would need lights → mediator identified as well as mediator → LST. What
can be convincing is an accounting: physics for mediator → LST, a descriptive lights → mediator,
and whether their product reproduces the TWFE (plan §6). P5 limits M1: ACAG PM2.5 carries almost
no cell-local variation at 10 km.

This notebook replaces `night_lst_mining_pm25.ipynb` (commit `a355ac2`), which used country ×
year FE, country clusters and kept flares. M1 re-runs its table in the fixed specification and
exports it in the same format (`output/tables/core_night_lst_mining_pm25_10km.*`).

Split from `02_core_analysis.ipynb` on 2026-09-30. The code cells and their outputs are from the 2026-09-29 run (SLURM job 24273525) and were **not re-run** after the split; only the text cells were relabelled.

In [1]:
import sys
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.analysis.spec import (   # the fixed specification (plan §2)
    PANEL, NTL_HI, COV_MIN, Y, Y_DAY, XLOG, Z, M, FE, FE_CTY, BASE,
    build_panel, fit, row, reg_table, first_stage_F,
    anderson_rubin_ci, coefplot,
)

build_panel()                     # cached: <data>/_analysis_panel_10km_v2.parquet
P = f"read_parquet('{PANEL}')"  # for DuckDB queries on the panel
sql = lambda q: duckdb.sql(q).df()


## Inputs: the TWFE and IV estimates the PM2.5 table reports

The same fits as B1 (`02_baseline.ipynb`) and I1 (`03_iv.ipynb`). The table in M1 puts the
TWFE, the 2SLS and the reduced forms next to the PM2.5 columns, so they are re-estimated here.

In [2]:
DAY = f"{Y_DAY} IS NOT NULL"
ladder = {
    "pooled OLS":              (f"{{y}} ~ {XLOG}", None),
    "+ pixel FE":              (f"{{y}} ~ {XLOG} | pixel_id", None),
    "TWFE (ctry x biome x yr)": (f"{{y}} ~ {XLOG} | {FE}", None),
    "TWFE (ctry x yr)":        (f"{{y}} ~ {XLOG} | {FE_CTY}", None),
    f"TWFE, cov_night >= {COV_MIN}": (f"{{y}} ~ {XLOG} | {FE}", f"cov_night >= {COV_MIN}"),
}
c1 = {}
for lab, (fml, sub) in ladder.items():
    c1[("night", lab)] = fit(fml.format(y=Y), subset=sub)
    c1[("day", lab)] = fit(fml.format(y=Y_DAY), subset=" AND ".join(filter(None, [DAY, sub])))
m_twfe = c1[("night", "TWFE (ctry x biome x yr)")]

c1_tab = pd.concat({out: reg_table([c1[(out, l)] for l in ladder], list(ladder), XLOG)
                    for out in ("night", "day")}, axis=1)
c1_tab

night                        day                  
                         estimate std_error       p estimate std_error       p
spec                                                                          
pooled OLS                -0.4008    0.2871  0.1628  -1.6203    0.3551  0.0000
+ pixel FE                 0.0253    0.0124  0.0422   0.0820    0.0221  0.0002
TWFE (ctry x biome x yr)  -0.0300    0.0056  0.0000  -0.0147    0.0123  0.2319
TWFE (ctry x yr)          -0.0265    0.0086  0.0020  -0.0151    0.0150  0.3139
TWFE, cov_night >= 0.75   -0.0373    0.0053  0.0000  -0.0305    0.0128  0.0172

In [4]:
fs      = fit(f"{XLOG} ~ {Z} | {FE}")
rf      = fit(f"{Y} ~ {Z} | {FE}")
m_iv    = fit(f"{Y} ~ 1 | {FE} | ({XLOG} ~ {Z})")
iv_disc = fit(f"{Y} ~ 1 | {FE} | (log_ntl_disc1 ~ {Z})", subset="log_ntl_disc1 IS NOT NULL")
rf_m1   = fit(f"ar_m1 ~ {Z} | {FE}")
rf_p1   = fit(f"ar_p1 ~ {Z} | {FE}")

ar_lo, ar_hi, ar_kind, ar_check = anderson_rubin_ci(rf, fs, rf_m1, rf_p1)
iv = row(m_iv, XLOG)
f = row(fs, Z)
print(f"first stage {f['est']:.4f} ({f['se']:.4f}), F = {(f['est'] / f['se']) ** 2:.1f}")
print(f"2SLS {iv['est']:+.4f} ({iv['se']:.4f})   Wald 95% CI [{iv['lo']:+.3f}, {iv['hi']:+.3f}]")
print(f"AR 95% set: {ar_kind} [{ar_lo:+.3f}, {ar_hi:+.3f}]   (check {ar_check:+.2e})")
pd.concat([reg_table([fs], ["first stage"], Z), reg_table([rf], ["reduced form (night LST)"], Z),
           reg_table([m_iv], ["2SLS own cell"], XLOG), reg_table([iv_disc], ["2SLS 3x3 disc"], "log_ntl_disc1")])

first stage 0.0642 (0.0076), F = 71.0
2SLS -0.0259 (0.1503)   Wald 95% CI [-0.321, +0.269]
AR 95% set: bounded [-0.322, +0.284]   (check +1.12e-12)


,estimate,std_error,p
spec,,,
first stage,0.0642,0.0076,0.0000
reduced form (night LST),-0.0017,0.0097,0.8633
2SLS own cell,-0.0259,0.1503,0.8630
2SLS 3x3 disc,-0.0300,0.1737,0.8629


## M1. The PM2.5 channel

On the pixel-years with PM2.5, separately for night and day LST:

- short: `Y ~ X | FE`;
- long: `Y ~ X + pm25 | FE`, with δ = the PM2.5 coefficient;
- auxiliary: `pm25 ~ X | FE`, with γ = the lights coefficient.

The omitted-variable identity β_short − β_long = γ·δ holds exactly in-sample. `identity_gap`
checks it. This uses PM2.5 as a descriptive decomposition of the OLS association. It does not
identify a causal direct effect.

**Reading rules** (plan §6, ◇):

- γ not significantly > 0: the control *cannot* move β. "No change" then says ACAG sees no
  lights-linked PM2.5 change, not that pollution is not the channel.
- δ ≥ 0: within a cell, more PM2.5 goes with warmer LST, and the aerosol-cooling premise fails
  for this measure and outcome.
- γ > 0 and δ < 0, but |γ·δ| < ¼ |β_short|: PM2.5, as ACAG measures it, explains little.
- Day vs night: aerosol dimming cools by day, and at night aerosols tend to warm. If the TWFE is
  more negative at night than by day, pollution is an unlikely sole explanation of the night
  coefficient.

In [6]:
PMS = f"{M} IS NOT NULL"
c4, acc = {}, {}
for out, y, sub in [("night", Y, PMS), ("day", Y_DAY, f"{PMS} AND {DAY}")]:
    s, l, a = (fit(f"{y} ~ {XLOG} | {FE}", subset=sub),
               fit(f"{y} ~ {XLOG} + {M} | {FE}", subset=sub),
               fit(f"{M} ~ {XLOG} | {FE}", subset=sub))
    c4[out] = dict(short=s, long=l, aux=a)
    bs, bl, g, d = row(s, XLOG), row(l, XLOG), row(a, XLOG), row(l, M)
    acc[out] = {"beta_short": bs["est"], "se_short": bs["se"], "beta_long": bl["est"], "se_long": bl["se"],
                "gamma (pm25 ~ X)": g["est"], "se_gamma": g["se"], "p_gamma": g["p"],
                "delta (Y ~ pm25 | X)": d["est"], "se_delta": d["se"], "p_delta": d["p"],
                "gamma*delta": g["est"] * d["est"], "beta_short - beta_long": bs["est"] - bl["est"],
                "identity_gap": (bs["est"] - bl["est"]) - g["est"] * d["est"],
                "share of beta_short via PM2.5": g["est"] * d["est"] / bs["est"]}
pd.DataFrame(acc).round(5)

,night,day
beta_short,-0.03005,-0.01467
se_short,0.00560,0.01227
beta_long,-0.02977,-0.01433
se_long,0.00554,0.01238
gamma (pm25 ~ X),-0.09816,-0.09810
se_gamma,0.06702,0.06712
p_gamma,0.14306,0.14387
delta (Y ~ pm25 | X),0.00282,0.00347
se_delta,0.00102,0.00240
p_delta,0.00586,0.14762


In [7]:
# The same channel on the IV side: 2SLS with the PM2.5 control, and PM2.5 on the instrument.
iv_pm = fit(f"{Y} ~ {M} | {FE} | ({XLOG} ~ {Z})", subset=PMS)
rf_pm = fit(f"{M} ~ {Z} | {FE}", subset=PMS)
pd.concat([reg_table([m_iv], ["2SLS, no control (full sample)"], XLOG),
           reg_table([iv_pm], ["2SLS + PM2.5 (PM2.5 sample)"], XLOG),
           reg_table([iv_pm], ["  PM2.5 coefficient in that 2SLS"], M),
           reg_table([rf_pm], ["reduced form PM2.5 ~ mines"], Z)])

,estimate,std_error,p
spec,,,
"2SLS, no control (full sample)",-0.0259,0.1503,0.8630
2SLS + PM2.5 (PM2.5 sample),-0.0204,0.1507,0.8925
PM2.5 coefficient in that 2SLS,0.0028,0.0010,0.0058
reduced form PM2.5 ~ mines,-0.1263,0.0667,0.0582


In [8]:
# The night_lst_mining_pm25 table, in the fixed specification. Same layout and export format.
def star(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .1 else ""

def cell(m, term, d=4):
    r = row(m, term)
    return f"{r['est']:.{d}f}{star(r['p'])}", f"({r['se']:.{d}f})"

def nobs(subset=None):
    w = " AND ".join(filter(None, [BASE, subset]))
    return duckdb.sql(f"SELECT count(*), count(DISTINCT pixel_id), count(DISTINCT GID_1), min(year), max(year) "
                      f"FROM {P} WHERE {w}").fetchone()

S_MAIN, S_PM = nobs(), nobs(PMS)
cols = ["(1) TWFE", "(2) 2SLS", "(3) RF: Temp", "(4) TWFE + PM2.5", "(5) RF: PM2.5"]
fits = dict(c1=m_twfe, c2=m_iv, c3=rf, c4=c4["night"]["long"], c5=rf_pm, fs=fs)
rows = {}
def put(label, entries):
    b = dict.fromkeys(cols, ""); s = dict.fromkeys(cols, "")
    for c, (m, term) in entries.items():
        b[c], s[c] = cell(fits[m], term)
    rows[label] = b; rows[label + " se"] = s

put("log(1+NTL)",         {cols[0]: ("c1", XLOG), cols[1]: ("c2", XLOG), cols[3]: ("c4", XLOG)})
put("Mines within 20 km", {cols[2]: ("c3", Z), cols[4]: ("c5", Z)})
put("PM2.5",              {cols[3]: ("c4", M)})
put("First stage: Mines 20 km -> log(1+NTL)", {cols[1]: ("fs", Z)})
rows["First-stage F (cluster-robust)"] = {**dict.fromkeys(cols, ""), cols[1]: f"{(f['est'] / f['se']) ** 2:.1f}"}
rows["Anderson-Rubin 95% CI"] = {**dict.fromkeys(cols, ""), cols[1]: f"[{ar_lo:.3f}, {ar_hi:.3f}]"}
samp = lambda i: S_PM if i >= 3 else S_MAIN
rows["Dep. var."]                    = dict(zip(cols, ["Night LST"] * 4 + ["PM2.5"]))
rows["Pixel FE"]                     = dict.fromkeys(cols, "Yes")
rows["Country x biome x year FE"]    = dict.fromkeys(cols, "Yes")
rows["Observations"]                 = {c: f"{samp(i)[0]:,}" for i, c in enumerate(cols)}
rows["Pixels"]                       = {c: f"{samp(i)[1]:,}" for i, c in enumerate(cols)}
rows["ADM1 regions (clusters)"]      = {c: f"{samp(i)[2]:,}" for i, c in enumerate(cols)}
rows["Years"]                        = {c: f"{samp(i)[3]}-{samp(i)[4]}" for i, c in enumerate(cols)}
tab = pd.DataFrame(rows).T[cols]
tab.index = ["" if i.endswith(" se") else i for i in tab.index]

bs = acc["night"]
note = ("SE clustered by ADM1 in parentheses. * p<0.1, ** p<0.05, *** p<0.01. 10 km grid, gas-flare "
        "pixel-years dropped. Cols (4)-(5) restricted to pixel-years with PM2.5.\n"
        f"Col (1) on the PM2.5 sample: {bs['beta_short']:.4f} ({bs['se_short']:.4f}). "
        f"PM2.5 on log(1+NTL) on that sample (gamma): {bs['gamma (pm25 ~ X)']:.4f} ({bs['se_gamma']:.4f}).")
OUT = PROJECT / "output" / "tables"
OUT.mkdir(parents=True, exist_ok=True)
tab.to_csv(OUT / "core_night_lst_mining_pm25_10km.csv")
md_rows = ["| | " + " | ".join(cols) + " |", "|---" * (len(cols) + 1) + "|"]
md_rows += ["| " + " | ".join([str(i)] + list(r)) + " |" for i, r in zip(tab.index, tab.values)]
(OUT / "core_night_lst_mining_pm25_10km.md").write_text("\n".join(md_rows) + "\n\n" + note + "\n")
(OUT / "core_night_lst_mining_pm25_10km.tex").write_text(tab.to_latex(escape=True))
print(note)
tab

SE clustered by ADM1 in parentheses. * p<0.1, ** p<0.05, *** p<0.01. 10 km grid, gas-flare pixel-years dropped. Cols (4)-(5) restricted to pixel-years with PM2.5.
Col (1) on the PM2.5 sample: -0.0300 (0.0056). PM2.5 on log(1+NTL) on that sample (gamma): -0.0982 (0.0670).


,(1) TWFE,(2) 2SLS,(3) RF: Temp,(4) TWFE + PM2.5,(5) RF: PM2.5
log(1+NTL),-0.0300***,-0.0259,,-0.0298***,
,(0.0056),(0.1503),,(0.0055),
Mines within 20 km,,,-0.0017,,-0.1263*
,,,(0.0097),,(0.0667)
PM2.5,,,,0.0028***,
,,,,(0.0010),
First stage: Mines 20 km -> log(1+NTL),,0.0642***,,,
,,(0.0076),,,
First-stage F (cluster-robust),,71.0,,,
Anderson-Rubin 95% CI,,"[-0.322, 0.284]",,,


**Result.** On the PM2.5 sample (all but ~70 pixel-years):

| | night | day |
|---|---:|---:|
| β_short → β_long | −0.0300 → −0.0298 | −0.0147 → −0.0143 |
| γ (PM2.5 on lights) | −0.098 (0.067), p = 0.14 | −0.098 (0.067), p = 0.14 |
| δ (LST on PM2.5 given lights) | +0.0028 (0.0010), p = 0.006 | +0.0035 (0.0024), p = 0.15 |
| γ·δ = β_short − β_long | −0.0003 | −0.0003 |
| share of β_short via PM2.5 | 0.9 % | 2.3 % |

The identity holds exactly (gap 0). On the IV side, 2SLS with the PM2.5 control is −0.020 (0.151).
PM2.5 on mines is −0.126 (0.067), p = 0.06.

**Interpretation.** Three rules fire, and together with P5 they settle how to read "no change":

- ◇ **γ is not significantly positive** (it is negative). Within a cell, lights do not raise
  ACAG PM2.5, so the control *could not* have moved β. The unchanged coefficient reflects that,
  not the absence of a pollution channel.
- ◇ **δ ≥ 0**, significantly so at night. More ACAG PM2.5 goes with warmer LST, the opposite of
  the aerosol-cooling premise, for this measure and outcome. At night that fits the physics
  (aerosols trap longwave); by day δ is also positive but imprecise.
- ◇ **The TWFE is more negative at night than by day** (−0.030 vs −0.015), where daytime dimming
  should dominate. Pollution is unlikely to be the sole explanation of the night coefficient.
- P5 ◇: ACAG carries almost no cell-local variation at 10 km (r² 0.97 with its neighbours), so
  a local aerosol channel is outside what this control can measure in the first place.

The fixed specification reproduces the reading of `night_lst_mining_pm25.ipynb` (TWFE −0.023,
γ ≈ −0.19 there), with ADM1 clusters, biome × year FE and flares dropped.

## Summary

| Block | Result | ◇ |
|---|---|---|
| M1 | γ = −0.10 (n.s.), δ = +0.003 (night, p = 0.006); PM2.5 accounts for 1–2 % of β | **the control could not move β; δ ≥ 0; night more negative than day** |

"Controlling for PM2.5 does not move the TWFE coefficient" holds, but the outcome was fixed
before the test: ACAG PM2.5 does not rise with lights within a cell (γ ≤ 0), it enters with a
positive sign (δ > 0), and at 10 km it carries almost no local variation (P5). The test can rule
out a *regional* ACAG-measured aerosol explanation of the night coefficient, not a local
pollution channel. Which channels could still fit the negative night sign is in plan §6; the
land-cover accounting is worth building only if Story 1 finds a robust association.